# DENTEX dental charting with YOLOv8
YOLOv8s tooth numbering (32 FDI classes) + diagnosis (caries, deep caries, periapical lesion, impacted), CLAHE preprocessing, findings mapped to teeth, ONNX export.

Settings: GPU T4 x2, Internet on, dataset `dentex` attached.

In [ ]:
# 0) Install packages
!pip -q install -U ultralytics onnx onnxslim onnxruntime

In [ ]:
# 1) Setup: config, GPU check, find the DENTEX annotation files
import os, json, glob, random, csv, subprocess, time, shutil
from collections import defaultdict, Counter
import cv2, numpy as np, torch

W = "/kaggle/working"
EPOCHS = 80
IMGSZ = 1024
FDI = [f"{q}{t}" for q in range(1, 5) for t in range(1, 9)]  # 32 FDI teeth: 11..48
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
for d in ("results", "logs", "models", "demo"):
    os.makedirs(f"{W}/{d}", exist_ok=True)

NGPU = torch.cuda.device_count()
print("GPUs:", NGPU, [torch.cuda.get_device_name(i) for i in range(NGPU)])
assert NGPU > 0, "No GPU: Settings > Accelerator > GPU T4 x2"

def find_json(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found under /kaggle/input"
    return hits[0]

J_TEETH = find_json("train_quadrant_enumeration.json")
J_DIAG = find_json("train_quadrant_enumeration_disease.json")
print(J_TEETH)
print(J_DIAG)

In [ ]:
# 2) Data prep: CLAHE + COCO -> YOLO + 85/15 split
#    teeth: 32 FDI classes (quadrant-enumeration set)
#    diag : 4 diagnosis classes (quadrant-enumeration-disease set), tooth number kept for evaluation
def build(json_path, out, task):
    d = json.load(open(json_path))
    src = os.path.dirname(json_path)
    on_disk = {os.path.basename(p): p for p in glob.glob(f"{src}/**/*.*", recursive=True)
               if p.lower().endswith((".png", ".jpg", ".jpeg"))}
    quad = {c["id"]: str(c["name"]) for c in d["categories_1"]}
    enum = {c["id"]: str(c["name"]) for c in d["categories_2"]}
    if task == "diag":
        cats = sorted(d["categories_3"], key=lambda c: c["id"])
        names = [str(c["name"]) for c in cats]
        to_cls = {c["id"]: i for i, c in enumerate(cats)}
    else:
        names = FDI
    by_img = defaultdict(list)
    for a in d["annotations"]:
        by_img[a["image_id"]].append(a)
    ims = [im for im in d["images"] if os.path.basename(im["file_name"]) in on_disk]
    random.Random(42).shuffle(ims)
    nval = int(len(ims) * 0.15)
    splits = {"val": ims[:nval], "train": ims[nval:]}
    gt, cnt = [], {s: Counter() for s in splits}
    for s, lst in splits.items():
        os.makedirs(f"{out}/images/{s}", exist_ok=True)
        os.makedirs(f"{out}/labels/{s}", exist_ok=True)
        for im in lst:
            fn = os.path.basename(im["file_name"])
            stem = os.path.splitext(fn)[0]
            Wd, Ht = im["width"], im["height"]
            dst = f"{out}/images/{s}/{stem}.png"
            if not os.path.exists(dst):
                g = cv2.imread(on_disk[fn], cv2.IMREAD_GRAYSCALE)
                cv2.imwrite(dst, clahe.apply(g))
            lines = []
            for a in by_img[im["id"]]:
                x, y, w, h = a["bbox"]
                x, y = max(0, x), max(0, y)
                w, h = min(w, Wd - x), min(h, Ht - y)
                if w <= 1 or h <= 1:
                    continue
                fdi = f"{quad[a['category_id_1']]}{enum[a['category_id_2']]}"
                cls = to_cls[a["category_id_3"]] if task == "diag" else FDI.index(fdi)
                lines.append(f"{cls} {(x + w/2)/Wd:.6f} {(y + h/2)/Ht:.6f} {w/Wd:.6f} {h/Ht:.6f}")
                cnt[s][names[cls]] += 1
                gt.append(dict(split=s, image=stem, cls=names[cls], fdi=fdi, x=x, y=y, w=w, h=h))
            with open(f"{out}/labels/{s}/{stem}.txt", "w") as f:
                f.write("\n".join(lines))
    with open(f"{out}/data.yaml", "w") as f:
        f.write(f"path: {out}\ntrain: images/train\nval: images/val\nnc: {len(names)}\nnames: {names}\n")
    for s in splits:
        info = dict(cnt[s]) if task == "diag" else f"{sum(cnt[s].values())} teeth"
        print(f"{task} {s}: {len(splits[s])} images | {info}")
    return names, gt

TEETH_NAMES, _ = build(J_TEETH, f"{W}/yolo_teeth", "teeth")
DIAG_NAMES, GT_DIAG = build(J_DIAG, f"{W}/yolo_diag", "diag")
json.dump(GT_DIAG, open(f"{W}/results/gt_diag.json", "w"))
print("diagnosis classes:", DIAG_NAMES)

In [ ]:
# 3) Train both YOLOv8s models in parallel (diag on GPU 0, teeth on GPU 1)
#    teeth model: horizontal flip OFF, a flip would swap left/right tooth numbers
from ultralytics import YOLO
os.chdir(W)
YOLO("yolov8s.pt")  # download weights once before the parallel runs

JOBS = [("diag", f"{W}/yolo_diag/data.yaml", []),
        ("teeth", f"{W}/yolo_teeth/data.yaml", ["fliplr=0.0"])]

def train_cmd(name, data, dev, extra):
    return ["yolo", "detect", "train", "model=yolov8s.pt", f"data={data}", f"imgsz={IMGSZ}",
            f"epochs={EPOCHS}", "batch=8", "patience=25", f"device={dev}", "workers=2",
            "cos_lr=True", "close_mosaic=10", f"project={W}/runs", f"name={name}", "exist_ok=True"] + extra

def last_line(path):
    try:
        txt = open(path, errors="ignore").read().replace("\r", "\n").strip().splitlines()
        prog = [l for l in txt if "/" in l and "G " in l]
        return (prog or txt or [""])[-1][:150]
    except Exception:
        return ""

t0 = time.time()
if NGPU >= 2:
    procs = []
    for dev, (name, data, extra) in enumerate(JOBS):
        lf = open(f"{W}/logs/{name}.log", "w")
        procs.append((name, subprocess.Popen(train_cmd(name, data, dev, extra), stdout=lf, stderr=subprocess.STDOUT, cwd=W)))
        print(f"started {name} on GPU {dev}", flush=True)
        time.sleep(120)  # stagger startup checks/downloads
    while any(p.poll() is None for _, p in procs):
        time.sleep(300)
        for name, p in procs:
            state = "running" if p.poll() is None else f"done ({p.returncode})"
            print(f"[{(time.time()-t0)/60:5.1f} min] {name}: {state} | {last_line(f'{W}/logs/{name}.log')}", flush=True)
else:
    for name, data, extra in JOBS:
        print(f"training {name} on GPU 0", flush=True)
        with open(f"{W}/logs/{name}.log", "w") as lf:
            subprocess.run(train_cmd(name, data, 0, extra), stdout=lf, stderr=subprocess.STDOUT, cwd=W)

for name, _, _ in JOBS:
    best = f"{W}/runs/{name}/weights/best.pt"
    if not os.path.exists(best):
        print(open(f"{W}/logs/{name}.log", errors="ignore").read()[-3000:])
        raise RuntimeError(f"{name} training failed, log above")
    shutil.copy(best, f"{W}/models/{name}_yolov8s_best.pt")
print(f"Training finished in {(time.time()-t0)/60:.1f} min")

In [ ]:
# 4) Evaluate: per-class precision / recall / F1 / mAP on the validation split
def evaluate(name, data):
    r = YOLO(f"{W}/runs/{name}/weights/best.pt").val(data=data, imgsz=IMGSZ, batch=8, device=0,
                                                       project=f"{W}/runs", name=f"{name}_val", exist_ok=True, verbose=False)
    rows = []
    for i, c in enumerate(r.box.ap_class_index):
        p, rc = float(r.box.p[i]), float(r.box.r[i])
        rows.append([r.names[int(c)], p, rc, 2*p*rc/(p+rc+1e-9), float(r.box.ap50[i]), float(r.box.ap[i])])
    P, R = float(r.box.mp), float(r.box.mr)
    rows.append(["ALL", P, R, 2*P*R/(P+R+1e-9), float(r.box.map50), float(r.box.map)])
    with open(f"{W}/results/{name}_metrics.csv", "w", newline="") as f:
        csv.writer(f).writerows([["class", "precision", "recall", "f1", "mAP50", "mAP50-95"]] +
                                [[x[0]] + [round(v, 3) for v in x[1:]] for x in rows])
    print(f"\n=== {name} (validation) ===")
    print(f"{'class':20s} {'P':>6s} {'R':>6s} {'F1':>6s} {'mAP50':>6s} {'mAP50-95':>8s}")
    show = rows if name == "diag" else sorted(rows[:-1], key=lambda x: x[2])[:6] + [rows[-1]]
    if name != "diag":
        print("(6 teeth with the lowest recall, then overall)")
    for x in show:
        print(f"{x[0]:20s} {x[1]:6.3f} {x[2]:6.3f} {x[3]:6.3f} {x[4]:6.3f} {x[5]:8.3f}")

for name, data, _ in JOBS:
    evaluate(name, data)

In [ ]:
# 5) Dental charting: map each finding to its FDI tooth, check tooth assignment, save overlays
def assign_tooth(box, teeth):
    # pick the tooth box that covers the largest share of the finding box
    x1, y1, x2, y2 = box
    area = max(1e-6, (x2 - x1) * (y2 - y1))
    best, score = None, 0.0
    for fdi, (a1, b1, a2, b2) in teeth:
        inter = max(0, min(x2, a2) - max(x1, a1)) * max(0, min(y2, b2) - max(y1, b1))
        if inter / area > score:
            best, score = fdi, inter / area
    return best, score

tm = YOLO(f"{W}/models/teeth_yolov8s_best.pt")
dm = YOLO(f"{W}/models/diag_yolov8s_best.pt")
gt_by = defaultdict(list)
for g in GT_DIAG:
    if g["split"] == "val":
        gt_by[g["image"]].append(g)
COLORS = [(0, 0, 255), (0, 165, 255), (255, 0, 255), (0, 255, 255), (255, 128, 0)]
charts, ok, tot = {}, 0, 0

for k, p in enumerate(sorted(glob.glob(f"{W}/yolo_diag/images/val/*.png"))):
    stem = os.path.splitext(os.path.basename(p))[0]
    tr = tm.predict(p, imgsz=IMGSZ, conf=0.25, verbose=False)[0]
    teeth = [(tr.names[int(c)], b.tolist()) for c, b in zip(tr.boxes.cls, tr.boxes.xyxy)]
    dr = dm.predict(p, imgsz=IMGSZ, conf=0.25, verbose=False)[0]
    finds = []
    for c, b, s in zip(dr.boxes.cls, dr.boxes.xyxy, dr.boxes.conf):
        fdi, _ = assign_tooth(b.tolist(), teeth)
        finds.append(dict(tooth=fdi, diagnosis=dr.names[int(c)], confidence=round(float(s), 3),
                          box=[round(v) for v in b.tolist()]))
    charts[stem] = sorted(finds, key=lambda f: f["tooth"] or "99")
    for g in gt_by[stem]:  # does a ground-truth finding land on the right tooth?
        fdi, _ = assign_tooth([g["x"], g["y"], g["x"] + g["w"], g["y"] + g["h"]], teeth)
        tot += 1
        ok += int(fdi == g["fdi"])
    if k < 15:
        img = cv2.cvtColor(cv2.imread(p, cv2.IMREAD_GRAYSCALE), cv2.COLOR_GRAY2BGR)
        for fdi, (a1, b1, a2, b2) in teeth:
            cv2.rectangle(img, (int(a1), int(b1)), (int(a2), int(b2)), (90, 200, 90), 2)
            cv2.putText(img, fdi, (int(a1) + 3, int(b1) + 26), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (90, 200, 90), 2)
        for f in finds:
            x1, y1, x2, y2 = f["box"]
            col = COLORS[list(dm.names.values()).index(f["diagnosis"]) % len(COLORS)]
            cv2.rectangle(img, (x1, y1), (x2, y2), col, 4)
            cv2.putText(img, f"{f['tooth']} {f['diagnosis']} {f['confidence']:.2f}", (x1, max(30, y1 - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, col, 3)
        cv2.imwrite(f"{W}/demo/{stem}_chart.jpg", img)

json.dump(charts, open(f"{W}/results/dental_charts_val.json", "w"), indent=1)
print(f"Tooth assignment accuracy on ground-truth findings: {ok/max(1, tot):.3f} ({ok}/{tot})")
ex = next((s for s in charts if charts[s]), None)
if ex:
    print(f"\nExample chart for {ex}:")
    for f in charts[ex]:
        print(f"  Tooth {f['tooth']}: {f['diagnosis']} ({f['confidence']:.2f})")

In [ ]:
# 6) Export both models to ONNX + sanity check with ONNX Runtime
for name in ("teeth", "diag"):
    out = YOLO(f"{W}/models/{name}_yolov8s_best.pt").export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True)
    dst = f"{W}/models/{name}_yolov8s.onnx"
    shutil.copy(out, dst)
    print(f"{name}: {dst} ({os.path.getsize(dst)/1e6:.1f} MB)")

sample = sorted(glob.glob(f"{W}/yolo_diag/images/val/*.png"))[0]
for name in ("teeth", "diag"):
    r = YOLO(f"{W}/models/{name}_yolov8s.onnx", task="detect").predict(sample, imgsz=IMGSZ, device="cpu", verbose=False)[0]
    print(f"ONNX {name} OK: {len(r.boxes)} detections on {os.path.basename(sample)}")

In [ ]:
# 7) Show a few charted X-rays and clean up the prepared image copies (keeps output small)
import matplotlib.pyplot as plt
for p in sorted(glob.glob(f"{W}/demo/*.jpg"))[:3]:
    plt.figure(figsize=(18, 9))
    plt.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(os.path.basename(p))
    plt.show()

for d in ("yolo_teeth/images", "yolo_diag/images"):
    shutil.rmtree(f"{W}/{d}", ignore_errors=True)
print("Outputs:", sorted(os.listdir(f"{W}/models")), sorted(os.listdir(f"{W}/results")))